<a href="https://colab.research.google.com/github/N3ma-Rahim/SpaceSoil/blob/main/spacesoil_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpaceSoil — Unified Spatiotemporal Fusion + Forecasting Pipeline
**Regions:** Mansoura, Fayoum, Ismailia, Luxor, Marsa Matrouh  |  **Period:** 2020–2025  |  **Cadence:** MODIS MOD13Q1 16-day composites

**Sources:** NASA POWER (daily weather) · MODIS MOD13Q1 (NDVI/EVI) · NASA SMAP L4 (soil moisture) · SoilGrids (static soil)

## Audit findings (what was wrong in the uploaded files, and what this notebook does about it)

| # | Source | Finding | Fix in this notebook |
|---|--------|---------|----------------------|
| 1 | Integration | `integrate_datasets.ipynb` imports `integrate_datasets.py` and reads `DATA_INTEGRATION_README.md` — **neither was uploaded**, so it cannot run. It also expects `SpaceSoil_Data/modis_cleaned.csv`, which the MODIS notebook **never exports**, and has **no SMAP input**. | Fusion rewritten from scratch (Section 5). |
| 2 | SMAP | Extraction covers **2 days** (2024-12-31 → 2025-01-01); `START_DATE/END_DATE` are test values. Leftover `.qa` / `.iso.xml` files are never deleted. The Harmony branch repeats the same 2-day window. | Resumable multi-year extractor with checkpointing and cleanup (Section 2.3). |
| 3 | SoilGrids | The exported 0–30 cm summary is **all zeros / NaN**. The center pixel returned `0` (the soil mask value for urban pixels at city-center coordinates), and `valid = arr != -32768` accepted it. The notebook's own validation printed 3 FAILs, but the CSV was exported anyway. | Detects invalid soil table; re-reads tiles treating `0` as masked and widening the search radius (Section 2.4). |
| 4 | MODIS | Only `Good` kept → **Marsa Matrouh keeps 2 of 139 composites** (122 are `Marginal`). Lags were computed *after* dropping rows, so `shift(1)` crosses multi-cycle gaps. Seasonal baseline / anomaly used **all years incl. 2025** (test leakage). No cleaned CSV is exported. Row for 2019-12-19 lies outside the weather period. | Keep reliability 0 and 1, drop 3 (cloudy) and 2 (snow). Complete 16-day grid so lags/targets mean "previous/next cycle". Train-only climatology. |
| 5 | MODIS | Mean NDVI is ~0.13 (max ~0.33). That is **urban/bare-soil level**, not cropland: the points are city centers. | Flagged in the audit; see the warning printed in Section 3. |
| 6 | POWER | The notebook references `monthly_df` / `year_month`, which are created in no cell (non-reproducible). Exported CSV still carries `year_month`; stale cell outputs list rolling columns that were dropped. Metadata says `ALLSKY_SFC_SW_DWN` is kWh/m²/day, but POWER AG returns **MJ/m²/day** (January values of 9–11 are impossible in kWh). `T2M_MIN` missing from metadata. | Loader keeps the needed columns only; unit documented as MJ/m²/day. |
| 7 | Schemas | `region/date` (POWER, SMAP, SoilGrids) vs `Region/Date` (MODIS) vs `Category/Date` (raw AppEEARS); `Marsa_Matrouh` vs `Marsa Matrouh`; four different output folders. | One normalizer + a recursive file finder. |
| 8 | Time alignment | MODIS `Date` is the **composite start**; the composite covers `Date … Date+15`. A "16 days ending on the MODIS date" window is the *previous* composite's period. | Weather/SMAP are aggregated over exactly the composite span; the forecast origin is the composite end. |

## Design decisions
* **Forecast origin = end of composite *k*.** Features use only cycles ≤ *k*; targets are NDVI and root-zone SM of cycle *k+1*.
* **Split by target date.** Train: target cycle in 2020–2024. Test: origin and target in 2025. The origin in Dec 2024 (target in Jan 2025) is dropped as an embargo.
* **Models predict the change from the current value** (`next − current`), and every result is reported against persistence and climatology baselines.
* **Demo mode.** `DEMO_MODE=True` runs everything on synthetic data with the *same schemas*. It verifies the code path only; **its metrics are not scientific results.**

In [1]:
import os, sys, json, glob, math, time, random, subprocess, warnings, shutil, importlib
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.width", 200, "display.max_columns", 60)


def ensure(pip_name, import_name=None):
    """Import a package, pip-installing it only if it is missing."""
    name = import_name or pip_name
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        return importlib.import_module(name)


ensure("xgboost")
ensure("scikit-learn", "sklearn")
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("pandas", pd.__version__, "| numpy", np.__version__, "| torch", torch.__version__, "| xgboost", xgb.__version__)

pandas 2.2.3 | numpy 2.4.6 | torch 2.14.1+cu130 | xgboost 3.4.1


In [2]:
# ----------------------------- CONFIGURATION -----------------------------
DEMO_MODE = False          # True = run on synthetic data (code verification only)
if os.environ.get("SPACESOIL_DEMO") == "1":
    DEMO_MODE = True

START_DATE, END_DATE = "2020-01-01", "2025-12-31"
TEST_YEAR = 2025           # test: origin and target cycles inside this year
VAL_YEAR = 2024            # last training year, used only for early stopping
SEQ_LEN = 6                # GRU input length in 16-day cycles (~96 days)
N_SEEDS = 3                # GRU ensemble size
SEED = 42

MODIS_KEEP_RELIABILITY = (0, 1)   # 0 Good, 1 Marginal. 2 = snow/ice and 3 = cloudy are excluded.
NDVI_FFILL_LIMIT = 3              # causal gap fill, in cycles
MIN_POWER_FRAC = 0.8              # min fraction of window days with weather
MIN_SMAP_FRAC = 0.7               # min fraction of window days with SMAP

RUN_SMAP_EXTRACTION = True        # real mode: extract SMAP if the CSVs cover < 90% of days
SMAP_SAMPLE_START_HOURS = [9]     # UTC start hours of the 3-hourly granules sampled per day
RUN_SOIL_REPAIR = True            # real mode: re-derive SoilGrids if the table is zeros/NaN

OUT_DIR = Path("spacesoil_outputs"); OUT_DIR.mkdir(exist_ok=True)
DEMO_DIR = Path("spacesoil_demo_data")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not DEMO_MODE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

SEARCH_ROOTS = [str(DEMO_DIR)] if DEMO_MODE else [
    ".", "/content/drive/MyDrive/SpaceSoil", "/kaggle/input", "/kaggle/working", "/mnt/user-data/uploads"]

REGIONS = {  # (latitude, longitude) -- identical in all four extraction notebooks
    "Mansoura": (31.04, 31.38), "Fayoum": (29.31, 30.84), "Ismailia": (30.60, 32.27),
    "Luxor": (25.69, 32.64), "Marsa Matrouh": (31.35, 27.24)}
REGION_LIST = list(REGIONS)


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEMO_MODE:", DEMO_MODE, "| device:", DEVICE, "| search roots:", SEARCH_ROOTS)

Drive not mounted: Error: credential propagation was unsuccessful
DEMO_MODE: False | device: cuda | search roots: ['.', '/content/drive/MyDrive/SpaceSoil', '/kaggle/input', '/kaggle/working', '/mnt/user-data/uploads']


In [3]:
# ----------------------------- HELPERS -----------------------------
_REGION_ALIASES = {r.lower().replace("_", " ").strip(): r for r in REGIONS}


def norm_region(s):
    key = str(s).lower().replace("_", " ").strip()
    if key not in _REGION_ALIASES:
        raise ValueError(f"Unknown region label: {s!r}")
    return _REGION_ALIASES[key]


def find_files(patterns, roots=None):
    roots = roots or SEARCH_ROOTS
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for pat in patterns:
            hits += glob.glob(os.path.join(root, "**", pat), recursive=True)
    return sorted(set(hits), key=lambda p: -os.path.getmtime(p))


def sanity(df, name, date_col=None, key=None):
    """Standard sanity printout used after every major step."""
    print(f"\n[SANITY] {name}: shape={df.shape}")
    if date_col and date_col in df:
        print(f"  dates: {df[date_col].min()} -> {df[date_col].max()}")
    if key and key in df:
        print("  rows per", key + ":", df.groupby(key).size().to_dict())
    na = df.isna().sum()
    na = na[na > 0]
    print("  columns with NaN:", na.to_dict() if len(na) else "none")
    display(df.head(3))


def cycle_start_of(dates):
    """MODIS 16-day composite start for each date (composites start on DOY 1, 17, ..., 353)."""
    d = pd.to_datetime(pd.Series(dates))
    return d - pd.to_timedelta((d.dt.dayofyear - 1) % 16, unit="D")


def cycle_end_of(starts):
    s = pd.to_datetime(pd.Series(starts))
    year_end = pd.to_datetime(s.dt.year.astype(str) + "-12-31")
    return pd.concat([s + pd.Timedelta(days=15), year_end], axis=1).min(axis=1)


def make_cycle_grid(first_year, last_year):
    starts = []
    for y in range(first_year, last_year + 1):
        s = pd.Timestamp(f"{y}-01-01")
        while s.year == y:
            starts.append(s); s += pd.Timedelta(days=16)
    return pd.DataFrame({"cycle_start": starts})


_g = make_cycle_grid(2020, 2025)
assert len(_g) == 138 and _g.cycle_start.iloc[-1] == pd.Timestamp("2025-12-19"), "composite grid is wrong"
assert (cycle_start_of(_g.cycle_start) == _g.cycle_start).all(), "cycle_start_of disagrees with the grid"
print("Composite grid 2020-2025:", len(_g), "cycles per region; first", _g.cycle_start.iloc[0].date(), "last", _g.cycle_start.iloc[-1].date())

Composite grid 2020-2025: 138 cycles per region; first 2020-01-01 last 2025-12-19


## 1. Demo data generator (used only when `DEMO_MODE=True`)
Writes files with **exactly the same schemas** as the real extraction notebooks, so the loaders below are exercised.
The synthetic NDVI depends on the previous cycle's root-zone moisture, so a working pipeline should beat the baselines.

In [4]:
def make_demo_sources(out_dir, seed=7):
    rng = np.random.default_rng(seed)
    out = Path(out_dir); (out / "MODIS").mkdir(parents=True, exist_ok=True)
    days = pd.date_range("2020-01-01", "2025-12-31")
    n = len(days); doy = days.dayofyear.values
    season = np.sin(2 * np.pi * (doy - 110) / 365.25)
    params = {  # base temp, rain scale, irrigation boost, ndvi base
        "Mansoura": (21.5, 1.0, 0.020, 0.14), "Fayoum": (22.5, 0.3, 0.012, 0.13),
        "Ismailia": (22.0, 0.4, 0.010, 0.14), "Luxor": (26.0, 0.05, 0.008, 0.12),
        "Marsa Matrouh": (20.0, 0.8, 0.000, 0.10)}
    power_rows, smap_rows, modis_rows, soil_rows = [], [], [], []
    starts = [pd.Timestamp("2019-12-19")] + list(make_cycle_grid(2020, 2025).cycle_start)
    for r, (lat, lon) in REGIONS.items():
        base_t, rain_s, irr, nd0 = params[r]
        t = base_t + 8 * season + rng.normal(0, 1.2, n)
        tmax = t + 6 + rng.normal(0, .8, n); tmin = t - 6 + rng.normal(0, .8, n)
        rh = np.clip(60 - 15 * season + rng.normal(0, 6, n), 15, 95)
        rainy = rng.random(n) < (0.10 * rain_s * (1.2 - season))
        prec = rainy * rng.gamma(1.5, 4.0, n)
        ws = np.clip(3 + rng.normal(0, 1.0, n), 0.3, None)
        solar = np.clip(17 + 9 * season + rng.normal(0, 2.0, n), 3, None)  # MJ/m2/day
        s_rz = np.zeros(n); v = 0.12
        for i in range(n):
            v = v * 0.985 + 0.0035 * prec[i] * (0.6 + rain_s) + irr * 0.05 + 0.0006
            s_rz[i] = min(max(v, 0.03), 0.45)
        s_sf = np.clip(0.55 * s_rz + 0.01 * prec + rng.normal(0, 0.006, n), 0.01, 0.5)
        p = pd.DataFrame({"date": days, "T2M": t, "T2M_MAX": tmax, "T2M_MIN": tmin, "RH2M": rh,
                          "PRECTOTCORR": prec, "WS2M": ws, "ALLSKY_SFC_SW_DWN": solar,
                          "region": r, "latitude": lat, "longitude": lon})
        p["daily_temp_range"] = p.T2M_MAX - p.T2M_MIN
        p["year_month"] = p["date"].dt.strftime("%Y-%m")
        bad = rng.choice(n, 6, replace=False); p.loc[bad, "T2M"] = -999.0   # NASA fill flags
        power_rows.append(p)
        sm = pd.DataFrame({"region": r, "date": days, "sm_surface": s_sf, "sm_rootzone": s_rz,
                           "sm_profile": s_rz + 0.002, "sm_surface_wetness": (s_sf - .01) / .5,
                           "sm_rootzone_wetness": (s_rz - .03) / .42})
        smap_rows.append(sm)
        # MODIS: NDVI depends on mean root-zone moisture of the PREVIOUS cycle
        for k, st in enumerate(starts):
            prev = days[(days >= st - pd.Timedelta(days=16)) & (days < st)]
            prev_sm = s_rz[np.isin(days, prev)].mean() if len(prev) else 0.12
            ndvi = nd0 + 0.35 * (prev_sm - 0.1) + 0.02 * np.sin(2 * np.pi * (st.dayofyear - 60) / 365.25) + rng.normal(0, .004)
            u = rng.random()
            rel = (1 if u < .85 else 3) if r == "Marsa Matrouh" else (0 if u < .90 else (1 if u < .93 else 3))
            modis_rows.append({"Category": r, "ID": r.replace(" ", "_"), "Latitude": lat, "Longitude": lon,
                               "Date": st.strftime("%Y-%m-%d"), "MODIS_Tile": "h20v06",
                               "MOD13Q1_061__250m_16_days_EVI": ndvi * 0.6,
                               "MOD13Q1_061__250m_16_days_NDVI": ndvi,
                               "MOD13Q1_061__250m_16_days_VI_Quality": 2048,
                               "MOD13Q1_061__250m_16_days_pixel_reliability": rel})
        soil_rows.append({"region": r, "clay_0_30": 20 + 5 * rain_s, "sand_0_30": 45 - 6 * rain_s,
                          "silt_0_30": 35 + rng.normal(0, 1), "soc_0_30": 8 + 3 * irr * 100,
                          "phh2o_0_30": 7.9 + rng.normal(0, .1), "bdod_0_30": 1.35,
                          "nitrogen_0_30": 1.0 + rain_s, "cec_0_30": 18 + 3 * rain_s,
                          "wv0033_0_30": 28.0, "wv1500_0_30": 14.0, "radius_m_max": 1500.0})
    pd.concat(power_rows).to_csv(out / "nasa_power_cleaned.csv", index=False)
    pd.concat(smap_rows).to_csv(out / "smap_daily_5_regions.csv", index=False)
    pd.DataFrame(modis_rows).to_csv(out / "MODIS" / "SpaceSoil-MODIS-Vegetation-2020-2025-MOD13Q1-061-results.csv", index=False)
    pd.DataFrame(soil_rows).to_csv(out / "soilgrids_0_30cm_summary.csv", index=False)
    print("Demo sources written to", out)


if DEMO_MODE:
    make_demo_sources(DEMO_DIR)

## 2. Loaders (schema normalization + automatic recovery where a source is incomplete)
### 2.1 NASA POWER

In [5]:
POWER_VARS = ["T2M", "T2M_MAX", "T2M_MIN", "RH2M", "PRECTOTCORR", "WS2M", "ALLSKY_SFC_SW_DWN"]
POWER_RENAME = {"T2M": "t2m", "T2M_MAX": "t2m_max", "T2M_MIN": "t2m_min", "RH2M": "rh2m",
                "PRECTOTCORR": "prec", "WS2M": "ws2m", "ALLSKY_SFC_SW_DWN": "solar"}  # solar = MJ/m2/day


def fetch_power_api():
    """Fallback: download daily POWER (AG community) for the 5 points. No login needed."""
    import requests
    frames = []
    for r, (lat, lon) in REGIONS.items():
        resp = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=180, params={
            "parameters": ",".join(POWER_VARS), "community": "AG", "longitude": lon, "latitude": lat,
            "start": START_DATE.replace("-", ""), "end": END_DATE.replace("-", ""),
            "format": "JSON", "time-standard": "UTC"})
        resp.raise_for_status()
        d = pd.DataFrame(resp.json()["properties"]["parameter"])
        d.index = pd.to_datetime(d.index, format="%Y%m%d"); d.index.name = "date"
        d = d.reset_index(); d["region"] = r; d["latitude"] = lat; d["longitude"] = lon
        frames.append(d)
        print("POWER downloaded:", r, len(d))
    return pd.concat(frames, ignore_index=True)


def load_power():
    hits = find_files(["nasa_power_cleaned.csv"])
    if hits:
        print("POWER file:", hits[0])
        raw = pd.read_csv(hits[0])
    elif not DEMO_MODE:
        print("nasa_power_cleaned.csv not found -> downloading from the POWER API")
        raw = fetch_power_api()
    else:
        raise FileNotFoundError("POWER demo file missing")
    raw = raw.rename(columns={c: c.lower() for c in ("Region", "Date") if c in raw})
    raw["region"] = raw["region"].map(norm_region)
    raw["date"] = pd.to_datetime(raw["date"])
    raw[POWER_VARS] = raw[POWER_VARS].replace([-999, -999.0], np.nan)
    keep = ["region", "date", "latitude", "longitude"] + POWER_VARS
    df = raw[keep].drop_duplicates(["region", "date"]).rename(columns=POWER_RENAME)
    df = df.rename(columns={"latitude": "lat", "longitude": "lon"}).sort_values(["region", "date"]).reset_index(drop=True)
    return df


power_daily = load_power()
sanity(power_daily, "POWER daily (normalized)", "date", "region")

nasa_power_cleaned.csv not found -> downloading from the POWER API
POWER downloaded: Mansoura 2192
POWER downloaded: Fayoum 2192
POWER downloaded: Ismailia 2192
POWER downloaded: Luxor 2192
POWER downloaded: Marsa Matrouh 2192

[SANITY] POWER daily (normalized): shape=(10960, 11)
  dates: 2020-01-01 00:00:00 -> 2025-12-31 00:00:00
  rows per region: {'Fayoum': 2192, 'Ismailia': 2192, 'Luxor': 2192, 'Mansoura': 2192, 'Marsa Matrouh': 2192}
  columns with NaN: none


,region,date,lat,lon,t2m,t2m_max,t2m_min,rh2m,prec,ws2m,solar
0,Fayoum,2020-01-01,29.31,30.84,10.94,17.21,6.30,62.26,0.00,2.00,10.96
1,Fayoum,2020-01-02,29.31,30.84,10.38,15.79,5.94,63.20,0.53,2.98,11.64
2,Fayoum,2020-01-03,29.31,30.84,9.93,15.56,5.23,67.06,0.14,4.23,11.76


### 2.2 MODIS MOD13Q1 (raw AppEEARS CSV preferred, because the cleaned one is Good-only)

In [6]:
!pip install -q earthaccess pyproj pyhdf

import os
import earthaccess
import numpy as np
import pandas as pd
from datetime import datetime
from pyproj import Transformer
from pyhdf.SD import SD, SDC

# 1. تسجيل الدخول لـ NASA Earthdata
auth = earthaccess.login(strategy="interactive", persist=True)

# 2. إحداثيات الأراضي الزراعية المحدّثة
REGIONS_FARMLAND = {
    "Mansoura":      (31.0850, 31.4350),
    "Fayoum":        (29.3550, 30.8950),
    "Ismailia":      (30.5450, 32.1850),
    "Luxor":         (25.6450, 32.5950),
    "Marsa Matrouh": (31.2950, 27.1550),
}

# 3. البحث عن ملفات MOD13Q1.061
results = earthaccess.search_data(
    short_name="MOD13Q1",
    version="061",
    bounding_box=(26.5, 25.0, 33.5, 32.0),
    temporal=("2020-01-01", "2025-12-31"),
)
print(f"Found {len(results)} granules.")

# الإحداثيات الخاصة بـ MODIS Sinusoidal Projection
modis_sinu = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"
transformer = Transformer.from_crs("EPSG:4326", modis_sinu, always_xy=True)

def extract_modis_hdf4(filepath, regions_dict):
    records = []
    fname = os.path.basename(filepath)
    julian_str = fname.split(".")[1][1:]
    obs_date = datetime.strptime(julian_str, "%Y%j").strftime("%Y-%m-%d")

    # فتح ملف HDF4 وقراءة البيانات الوصفية والطبقات باستخدام pyhdf
    try:
        hdf = SD(filepath, SDC.READ)
    except Exception as e:
        print(f"Error opening {filepath}: {e}")
        return records

    # استخراج الطبقات الفرعية
    try:
        ndvi_obj = hdf.select("250m 16 days NDVI")
        evi_obj = hdf.select("250m 16 days EVI")
        rel_obj = hdf.select("250m 16 days pixel reliability")

        ndvi_arr = ndvi_obj.get()
        evi_arr = evi_obj.get()
        rel_arr = rel_obj.get()

        # قراءة خصائص الإسقاط الجغرافي للملف
        # نظام شبكة MODIS ISIN/SIN يغطي إحداثيات محددة بناءً على رقم الـ Tile المذكور في اسم الملف
        # بشكل مبسط، سنقوم بالتحويل الرياضي التقليدي لـ MODIS Sinusoidal
        # خطوط العرض والطول لـ Sinusoidal
        # r_earth = 6371007.181
        # سنعتمد على الحساب الجغرافي لمعرفة موقع البكسل
        # الحدود الافتراضية للـ Grid الخاصة بالـ Tile
        grid_attr = hdf.attributes().get("StructMetadata.0")
        # كبديل ديناميكي متين لتفادي التعقيد الرياضي لـ Sinusoidal بدون GDAL،
        # سنحدد الـ Tile بناءً على اسم الملف (مثل h20v05) لحساب زاوية البكسل بدقة:
        tile_str = fname.split(".")[2]
        h_tile = int(tile_str[1:3])
        v_tile = int(tile_str[4:6])

        # حساب حدود الـ Tile الجغرافية في نظام إسقاط Sinusoidal
        x_min = (h_tile - 18) * 1111950.519766666
        x_max = (h_tile - 17) * 1111950.519766666
        y_max = (9 - v_tile) * 1111950.519766666
        y_min = (8 - v_tile) * 1111950.519766666

        height, width = ndvi_arr.shape
        x_pixel_size = (x_max - x_min) / width
        y_pixel_size = (y_max - y_min) / height

        for reg_name, (lat, lon) in regions_dict.items():
            x, y = transformer.transform(lon, lat)
            if x_min <= x <= x_max and y_min <= y <= y_max:
                col = int((x - x_min) / x_pixel_size)
                row = int((y_max - y) / y_pixel_size)
                if 0 <= row < height and 0 <= col < width:
                    raw_ndvi = ndvi_arr[row, col]
                    raw_evi = evi_arr[row, col]
                    rel = int(rel_arr[row, col])

                    records.append({
                        "Region": reg_name,
                        "Date": obs_date,
                        "NDVI": raw_ndvi * 0.0001 if raw_ndvi != -3000 else np.nan,
                        "EVI":  raw_evi  * 0.0001 if raw_evi  != -3000 else np.nan,
                        "Pixel_Reliability": rel
                    })
    except Exception as e:
        print(f"Error processing variables in {filepath}: {e}")
    finally:
        hdf.end()

    return records

os.makedirs("modis_temp", exist_ok=True)
all_rows = []
batch_size = 30

for i in range(0, len(results), batch_size):
    batch = results[i:i+batch_size]
    downloaded_files = earthaccess.download(batch, local_path="modis_temp")
    for fp in downloaded_files:
        fp_str = str(fp)
        if fp_str.endswith(".hdf"):
            all_rows.extend(extract_modis_hdf4(fp_str, REGIONS_FARMLAND))
            os.remove(fp_str)
    print(f"Processed {min(i+batch_size, len(results))}/{len(results)} files...")

modis_df = pd.DataFrame(all_rows).drop_duplicates(subset=["Region", "Date"]).sort_values(["Region", "Date"])
modis_df.to_csv("modis_cleaned.csv", index=False)
print("✅ Done! Saved modis_cleaned.csv | Shape:", modis_df.shape)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
Enter your Earthdata Login username: manarsabry
Enter your Earthdata password: ··········
Found 417 granules.


QUEUEING TASKS | :   0%|          | 0/120 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/120 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/120 [00:00<?, ?it/s]

Processed 30/417 files...


QUEUEING TASKS | :   0%|          | 0/120 [00:00<?, ?it/s]

... (47 outputs left out)

QUEUEING TASKS | :   0%|          | 0/27 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/27 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/27 [00:00<?, ?it/s]

Processed 417/417 files...
✅ Done! Saved modis_cleaned.csv | Shape: (695, 5)


In [9]:
def load_modis():
    hits = find_files(["*MOD13Q1*results.csv"])
    cleaned = find_files(["modis_cleaned.csv"])
    if not hits and not cleaned:
        raise FileNotFoundError(
            "No MODIS file found. Place the raw AppEEARS CSV "
            "('SpaceSoil-MODIS-Vegetation-2020-2025-MOD13Q1-061-results.csv', produced by modis-dataset-nasa.ipynb) "
            "in the working directory or in Drive/SpaceSoil.")
    path = hits[0] if hits else cleaned[0]
    print("MODIS file:", path)
    raw = pd.read_csv(path)
    col = lambda suffix: next((c for c in raw.columns if c.endswith(suffix)), None)
    mapping = {"region": next((c for c in ("Category", "Region", "region") if c in raw), None),
               "date": next((c for c in ("Date", "date") if c in raw), None),
               "ndvi": col("_NDVI") or ("NDVI" if "NDVI" in raw else None),
               "evi": col("_EVI") or ("EVI" if "EVI" in raw else None),
               "reliability": col("_pixel_reliability") or ("Pixel_Reliability" if "Pixel_Reliability" in raw else None),
               "lat": next((c for c in ("Latitude", "latitude") if c in raw), None),
               "lon": next((c for c in ("Longitude", "longitude") if c in raw), None)}

    # If lat/lon are missing from the CSV, we will dynamically inject them based on region coordinates
    coord_fallback = False
    if mapping["lat"] is None or mapping["lon"] is None:
        coord_fallback = True
        mapping.pop("lat", None)
        mapping.pop("lon", None)

    missing = [k for k, v in mapping.items() if v is None]
    if missing:
        raise KeyError(f"MODIS columns not found for {missing}; columns present: {list(raw.columns)[:15]} ...")

    df = raw[[mapping[k] for k in mapping]].copy(); df.columns = list(mapping)
    df["region"] = df["region"].map(norm_region)

    if coord_fallback:
        # Map coords from REGIONS_FARMLAND if defined, otherwise REGIONS
        coords_dict = globals().get("REGIONS_FARMLAND", REGIONS)
        df["lat"] = df["region"].map(lambda r: coords_dict.get(r, (np.nan, np.nan))[0])
        df["lon"] = df["region"].map(lambda r: coords_dict.get(r, (np.nan, np.nan))[1])

    df["date"] = pd.to_datetime(df["date"])
    df["reliability"] = pd.to_numeric(df["reliability"], errors="coerce")
    for c in ("ndvi", "evi"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[(df[c] <= -0.2999) | (df[c] > 1.0), c] = np.nan
    return df.drop_duplicates(["region", "date"]).sort_values(["region", "date"]).reset_index(drop=True)


modis_raw = load_modis()
sanity(modis_raw, "MODIS raw (normalized)", "date", "region")
print("Reliability codes (0 Good, 1 Marginal, 2 Snow/Ice, 3 Cloudy):")
display(pd.crosstab(modis_raw.region, modis_raw.reliability))
strict_keep = modis_raw[modis_raw.reliability == 0].groupby("region").size()
policy_keep = modis_raw[modis_raw.reliability.isin(MODIS_KEEP_RELIABILITY)].groupby("region").size()
display(pd.DataFrame({"total": modis_raw.groupby("region").size(), "kept_Good_only": strict_keep,
                      f"kept_{MODIS_KEEP_RELIABILITY}": policy_keep}).fillna(0).astype(int))

MODIS file: ./modis_cleaned.csv

[SANITY] MODIS raw (normalized): shape=(695, 7)
  dates: 2019-12-19 00:00:00 -> 2025-12-19 00:00:00
  rows per region: {'Fayoum': 139, 'Ismailia': 139, 'Luxor': 139, 'Mansoura': 139, 'Marsa Matrouh': 139}
  columns with NaN: none


,region,date,ndvi,evi,reliability,lat,lon
0,Fayoum,2019-12-19,0.7692,0.5545,0,29.355,30.895
1,Fayoum,2020-01-01,0.7702,0.5894,0,29.355,30.895
2,Fayoum,2020-01-17,0.7613,0.6270,1,29.355,30.895


Reliability codes (0 Good, 1 Marginal, 2 Snow/Ice, 3 Cloudy):


reliability,0,1,3
region,,,
Fayoum,128,11,0
Ismailia,122,17,0
Luxor,137,1,1
Mansoura,134,5,0
Marsa Matrouh,134,1,4


,total,kept_Good_only,"kept_(0, 1)"
region,,,
Fayoum,139,128,139
Ismailia,139,122,139
Luxor,139,137,138
Mansoura,139,134,139
Marsa Matrouh,139,134,135


### 2.3 NASA SMAP L4 (SPL4SMGP v008): multi-year, resumable extraction
The original notebook downloaded **17 granules (2 days)** and every granule is a *global* file, so the full period is a long job.
This extractor: (1) searches quarter by quarter, (2) samples only the granule start hours in `SMAP_SAMPLE_START_HOURS` (default one per day),
(3) downloads in small batches, reads the 5 grid cells x 5 variables, **deletes every downloaded file** (`.h5`, `.qa`, `.iso.xml`),
(4) appends to a checkpoint CSV so an interrupted run resumes where it stopped.
Daily values are the mean of the sampled granules; the same hour is used every day, so any diurnal offset is constant.

In [ ]:
SMAP_SHORT_NAME, SMAP_VERSION = "SPL4SMGP", "008"
SMAP_VARS = ["sm_surface", "sm_rootzone", "sm_profile", "sm_surface_wetness", "sm_rootzone_wetness"]
SMAP_CELLS = pd.DataFrame({   # nearest EASE-2 9 km cells found in eda-nasa-datasets / smap_clean notebooks
    "region": ["Mansoura", "Fayoum", "Ismailia", "Luxor", "Marsa Matrouh"],
    "row": [393, 414, 398, 459, 389], "column": [2264, 2258, 2273, 2277, 2219]})
SMAP_DIR = Path("SMAP_Data"); (SMAP_DIR / "raw").mkdir(parents=True, exist_ok=True); (SMAP_DIR / "processed").mkdir(exist_ok=True)
SMAP_CKPT = SMAP_DIR / "processed" / "smap_granule_checkpoint_2020_2025.csv"
SMAP_FINAL = SMAP_DIR / "processed" / "smap_daily_2020_2025.csv"


def read_smap_cells(h5_path):
    """Read the 5 variables at the 5 fixed grid cells; fill value -9999 -> NaN."""
    h5py = ensure("h5py")
    rows = []
    with h5py.File(h5_path, "r") as h:
        for _, c in SMAP_CELLS.iterrows():
            rec = {"region": c["region"]}
            for v in SMAP_VARS:
                x = float(h[f"Geophysical_Data/{v}"][int(c["row"]), int(c["column"])])
                rec[v] = np.nan if x == -9999.0 else x
            rows.append(rec)
    return rows


def verify_smap_cells(h5_path, tol_deg=0.15):
    """Check that the hard-coded row/column pairs still point at the right place."""
    h5py = ensure("h5py")
    with h5py.File(h5_path, "r") as h:
        for _, c in SMAP_CELLS.iterrows():
            la = float(h["cell_lat"][int(c["row"]), int(c["column"])]); lo = float(h["cell_lon"][int(c["row"]), int(c["column"])])
            tl, tg = REGIONS[c["region"]]
            assert abs(la - tl) < tol_deg and abs(lo - tg) < tol_deg, f"SMAP cell for {c['region']} is off: ({la:.3f},{lo:.3f}) vs ({tl},{tg})"
    print("SMAP grid cells verified against region coordinates.")


def _smap_login(earthaccess):
    for strat in ("environment", "netrc", "interactive"):
        try:
            auth = earthaccess.login(strategy=strat, persist=(strat == "interactive"))
            if getattr(auth, "authenticated", False):
                print("Earthdata login OK via", strat); return auth
        except Exception as e:
            print("login", strat, "failed:", str(e)[:80])
    raise RuntimeError("Earthdata authentication failed")


def extract_smap_multi_year(start=START_DATE, end=END_DATE, hours=SMAP_SAMPLE_START_HOURS, batch=12, threads=8):
    earthaccess = ensure("earthaccess"); ensure("h5py")
    _smap_login(earthaccess)
    done = set(pd.read_csv(SMAP_CKPT)["file_key"].astype(str)) if SMAP_CKPT.exists() else set()
    print("Granules already in checkpoint:", len(done))
    qs = [q for q in pd.date_range(start, end, freq="QS") if q > pd.Timestamp(start)]
    quarters = [pd.Timestamp(start)] + qs + [pd.Timestamp(end) + pd.Timedelta(days=1)]
    t0, failed, verified = time.time(), [], False
    for qa, qb in zip(quarters[:-1], quarters[1:]):
        gr = earthaccess.search_data(short_name=SMAP_SHORT_NAME, version=SMAP_VERSION,
                                     temporal=(qa.strftime("%Y-%m-%d"), (qb - pd.Timedelta(seconds=1)).strftime("%Y-%m-%dT%H:%M:%S")))
        todo = []
        for g in gr:
            b = pd.Timestamp(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]).tz_localize(None)
            if b.hour not in hours:
                continue
            key = (b + pd.Timedelta(minutes=90)).strftime("%Y%m%dT%H%M%S")   # granule center time
            if key not in done and START_DATE <= key[:4] + "-" + key[4:6] + "-" + key[6:8] <= END_DATE:
                todo.append((key, g))
        print(f"{qa.date()}: {len(gr)} granules found, {len(todo)} to process | {(time.time()-t0)/60:.1f} min")
        for i in range(0, len(todo), batch):
            chunk = todo[i:i + batch]
            try:
                paths = earthaccess.download([g for _, g in chunk], local_path=str(SMAP_DIR / "raw"), threads=threads)
            except Exception as e:
                print("  download batch failed:", str(e)[:100]); failed += [k for k, _ in chunk]; continue
            rows = []
            try:
                for p in paths:
                    p = str(p)
                    if not p.endswith(".h5"):
                        continue
                    if not verified:
                        verify_smap_cells(p); verified = True
                    stamp = os.path.basename(p).split("_gph_")[1][:15]            # e.g. 20241231T223000
                    for rec in read_smap_cells(p):
                        rows.append({"file_key": stamp, "date": f"{stamp[:4]}-{stamp[4:6]}-{stamp[6:8]}", **rec})
            except Exception as e:
                print("  read failed:", type(e).__name__, str(e)[:100]); failed += [k for k, _ in chunk]
            finally:
                for p in paths:                                                 # remove .h5, .qa and .iso.xml
                    try: os.remove(str(p))
                    except OSError: pass
            if rows:
                pd.DataFrame(rows).to_csv(SMAP_CKPT, mode="a", header=not SMAP_CKPT.exists(), index=False)
                done |= {r["file_key"] for r in rows}
    print("Failed granule keys (re-run this cell to retry):", len(set(failed)))
    return aggregate_smap_checkpoint()


def aggregate_smap_checkpoint():
    ck = pd.read_csv(SMAP_CKPT).drop_duplicates(["file_key", "region"])
    ck["date"] = pd.to_datetime(ck["date"])
    daily = ck.groupby(["region", "date"], as_index=False).agg(**{v: (v, "mean") for v in SMAP_VARS}, n_granules=("file_key", "nunique"))
    daily.to_csv(SMAP_FINAL, index=False)
    print("SMAP daily saved:", SMAP_FINAL, daily.shape)
    return daily


def load_smap():
    n_days = len(pd.date_range(START_DATE, END_DATE))
    frames = []
    for p in find_files(["smap*daily*.csv"]):
        d = pd.read_csv(p); d["region"] = d["region"].map(norm_region); d["date"] = pd.to_datetime(d["date"])
        if "n_granules" not in d: d["n_granules"] = np.nan
        frames.append(d[["region", "date"] + SMAP_VARS + ["n_granules"]]); print("SMAP file:", p, d.shape)
    smap = (pd.concat(frames).drop_duplicates(["region", "date"], keep="first") if frames   # newest file wins
            else pd.DataFrame(columns=["region", "date"] + SMAP_VARS + ["n_granules"]))
    cov = smap[(smap.date >= START_DATE) & (smap.date <= END_DATE)].groupby("region").date.nunique() / n_days if len(smap) else pd.Series(dtype=float)
    cov = cov.reindex(REGION_LIST).fillna(0.0)
    print("SMAP day coverage per region:", cov.round(3).to_dict())
    if cov.min() < 0.9 and RUN_SMAP_EXTRACTION and not DEMO_MODE:
        print("SMAP covers < 90% of 2020-2025 -> running multi-year extraction (resumable; may take hours)")
        smap = extract_smap_multi_year()
        smap["region"] = smap["region"].map(norm_region)
        cov2 = smap.groupby("region").date.nunique() / n_days
        print("SMAP day coverage after extraction:", cov2.round(3).to_dict(), "(re-run this cell to resume if < 0.9)")
    elif cov.min() < 0.9:
        print("WARNING: SMAP coverage is incomplete; SMAP features will be NaN where missing.")
    smap = smap.sort_values(["region", "date"]).reset_index(drop=True)
    smap["date"] = pd.to_datetime(smap["date"])
    smap[SMAP_VARS] = smap[SMAP_VARS].replace(-9999.0, np.nan)
    return smap


smap_daily = load_smap()
sanity(smap_daily, "SMAP daily", "date", "region")

SMAP day coverage per region: {'Mansoura': 0.0, 'Fayoum': 0.0, 'Ismailia': 0.0, 'Luxor': 0.0, 'Marsa Matrouh': 0.0}
SMAP covers < 90% of 2020-2025 -> running multi-year extraction (resumable; may take hours)
Earthdata login OK via environment
Granules already in checkpoint: 0
2020-01-01: 729 granules found, 91 to process | 0.0 min


QUEUEING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/12 [00:00<?, ?it/s]

SMAP grid cells verified against region coordinates.


QUEUEING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

... (59 outputs left out)

PROCESSING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/12 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

### 2.4 SoilGrids: validate, and repair if the table is the all-zero export
Root cause: city-center points fall on the SoilGrids soil mask (urban pixels), where the WCS returns `0`, and `0` was treated as a valid value.
The repair re-reads the saved GeoTIFF tiles (or re-downloads them) with `0` and `-32768` treated as masked, widens the search radius
(center → 1.5 → 5 → 10 km) and recomputes the thickness-weighted 0–30 cm mean.

In [ ]:
SOIL_PROPS = ["clay", "sand", "silt", "soc", "phh2o", "bdod", "nitrogen", "cec", "wv0033", "wv1500"]
SOIL_DEPTHS = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}
SOIL_CONV = {"bdod": 100, "cec": 10, "clay": 10, "nitrogen": 100, "soc": 10, "phh2o": 10, "sand": 10, "silt": 10, "wv0033": 10, "wv1500": 10}
SOIL_RANGES = {"clay": (1, 100), "sand": (0.5, 100), "silt": (0.5, 100), "soc": (0.1, 600), "phh2o": (3, 10.5),
               "bdod": (0.5, 2.2), "nitrogen": (0.01, 50), "cec": (0.5, 150), "wv0033": (1, 70), "wv1500": (0.5, 60)}
SOIL_RADII = [0, 1500, 5000, 10000]
SOIL_HALF_WIDTH = 10000


def soil_table_status(soil):
    cols = [f"{p}_0_30" for p in ("clay", "sand", "silt")]
    if not all(c in soil for c in cols):
        return "invalid", "texture columns missing"
    vals = soil[cols]
    if vals.isna().all().all() or (vals.fillna(0) == 0).all().all():
        return "invalid", "texture is all zero/NaN (masked-pixel export)"
    tsum = vals.sum(axis=1)
    if not tsum.between(80, 120).all():
        return "warn", f"sand+silt+clay sums outside 80-120: {tsum.round(1).to_dict()}"
    return "ok", "texture sums plausible"


def soil_window_value(arr, valid, xs, ys, x, y):
    dist = np.hypot(xs - x, ys - y)
    ci = np.argmin(dist)
    for r in SOIL_RADII:
        if r == 0:
            mask = np.zeros(arr.shape, bool); mask.flat[ci] = True; mask &= valid
        else:
            mask = valid & (dist <= r)
        if mask.any():
            return float(np.median(arr[mask])), r
    return np.nan, np.nan


def read_soil_tile(path, x, y):
    rasterio = ensure("rasterio")
    from rasterio.transform import xy as raster_xy
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype("float64"); nodata = ds.nodata
        rows, cols = np.indices(arr.shape)
        xs, ys = raster_xy(ds.transform, rows.ravel(), cols.ravel(), offset="center")
        xs = np.array(xs).reshape(arr.shape); ys = np.array(ys).reshape(arr.shape)
    valid = (arr != -32768) & (arr != 0)          # <- the fix: 0 is the soil-mask value, not data
    if nodata is not None:
        valid &= arr != nodata
    return soil_window_value(arr, valid, xs, ys, x, y)


def fetch_soil_tile(prop, depth, region, x, y, tile_dir):
    import requests
    cid = f"{prop}_{depth}_Q0.5"; path = Path(tile_dir) / f"{cid}__{region.replace(' ', '_')}.tif"
    if path.exists():
        return path
    params = [("map", f"/map/{prop}.map"), ("SERVICE", "WCS"), ("VERSION", "2.0.1"), ("REQUEST", "GetCoverage"),
              ("COVERAGEID", cid), ("FORMAT", "GEOTIFF_INT16"),
              ("SUBSET", f"X({x - SOIL_HALF_WIDTH:.0f},{x + SOIL_HALF_WIDTH:.0f})"),
              ("SUBSET", f"Y({y - SOIL_HALF_WIDTH:.0f},{y + SOIL_HALF_WIDTH:.0f})"),
              ("SUBSETTINGCRS", "http://www.opengis.net/def/crs/EPSG/0/152160"),
              ("OUTPUTCRS", "http://www.opengis.net/def/crs/EPSG/0/152160")]
    last = ""
    for attempt in range(3):
        try:
            r = requests.get("https://maps.isric.org/mapserv", params=params, timeout=120)
            if r.status_code == 200 and r.content[:4] in (b"II*\x00", b"MM\x00*", b"II+\x00"):
                path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content); time.sleep(0.4); return path
            last = f"HTTP {r.status_code}"
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:100]}"
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(last)


def repair_soilgrids():
    pyproj = ensure("pyproj"); ensure("rasterio")
    tr = pyproj.Transformer.from_crs("EPSG:4326", "+proj=igh +datum=WGS84 +no_defs", always_xy=True)
    tdirs = [p for p in find_files(["raw_tiles"]) if os.path.isdir(p)]
    tile_dir = tdirs[0] if tdirs else "SoilGrids_Data/raw_tiles"
    print("SoilGrids tile directory:", tile_dir)
    rows = []
    for r, (lat, lon) in REGIONS.items():
        x, y = tr.transform(lon, lat); rec = {"region": r}; per_prop = {p: [] for p in SOIL_PROPS}; rmax = 0
        for p in SOIL_PROPS:
            for d, w in SOIL_DEPTHS.items():
                try:
                    v, rad = read_soil_tile(fetch_soil_tile(p, d, r, x, y, tile_dir), x, y)
                except Exception as e:
                    print(f"  {r} {p} {d}: {str(e)[:80]}"); v, rad = np.nan, np.nan
                v = v / SOIL_CONV[p] if pd.notna(v) else np.nan
                lo, hi = SOIL_RANGES[p]
                if pd.notna(v) and not (lo <= v <= hi): v = np.nan
                per_prop[p].append((v, w))
                if pd.notna(rad): rmax = max(rmax, rad)
        for p in SOIL_PROPS:
            ok = [(v, w) for v, w in per_prop[p] if pd.notna(v)]
            rec[f"{p}_0_30"] = np.average([v for v, _ in ok], weights=[w for _, w in ok]) if ok else np.nan
        rec["radius_m_max"] = rmax; rows.append(rec)
    out = pd.DataFrame(rows)
    out.to_csv("soilgrids_0_30cm_summary_REPAIRED.csv", index=False)
    print("Repaired soil table saved: soilgrids_0_30cm_summary_REPAIRED.csv")
    return out


def load_soil():
    hits = find_files(["soilgrids_0_30cm_summary*.csv"])
    soil = None
    if hits:
        # prefer a repaired file if present, otherwise the newest
        hits = sorted(hits, key=lambda p: ("REPAIRED" not in p, -os.path.getmtime(p)))
        print("SoilGrids file:", hits[0]); soil = pd.read_csv(hits[0])
        soil["region"] = soil["region"].map(norm_region)
    status, msg = soil_table_status(soil) if soil is not None else ("invalid", "file not found")
    print("SoilGrids status:", status, "-", msg)
    if status == "invalid" and RUN_SOIL_REPAIR and not DEMO_MODE:
        try:
            soil = repair_soilgrids(); status, msg = soil_table_status(soil); print("After repair:", status, "-", msg)
        except Exception as e:
            print("Soil repair failed:", type(e).__name__, str(e)[:150])
    if status == "invalid":
        print("WARNING: no valid SoilGrids data. Region one-hot columns will stand in as the static branch input.")
        return None, status
    num = [c for c in soil.columns if c.endswith("_0_30") and soil[c].dtype != object]
    soil = soil[["region"] + num + (["radius_m_max"] if "radius_m_max" in soil else [])].copy()
    soil[num] = soil[num].replace(0, np.nan) if status != "ok" else soil[num]
    return soil, status


soil_static, soil_status = load_soil()
if soil_static is not None:
    sanity(soil_static, "SoilGrids static", None, "region")

## 3. Automated audit (schema, coordinates, dates, coverage, quality)

In [ ]:
def run_audit():
    rows = []
    add = lambda name, status, detail: rows.append((status, name, detail))
    n_days = len(pd.date_range(START_DATE, END_DATE))
    grid_ok = lambda d: bool(((d.dayofyear - 1) % 16 == 0).all())

    for nm, df in (("POWER", power_daily), ("MODIS", modis_raw), ("SMAP", smap_daily)):
        regs = set(df.region.unique()) if len(df) else set()
        add(f"{nm}: 5 target regions present", "PASS" if regs == set(REGION_LIST) else "FAIL", f"missing={sorted(set(REGION_LIST) - regs)}")
    if soil_static is not None:
        add("SoilGrids: 5 target regions present", "PASS" if set(soil_static.region) == set(REGION_LIST) else "FAIL", "")

    # coordinates
    for nm, df, la, lo, tol in (("POWER", power_daily, "lat", "lon", 0.011), ("MODIS", modis_raw, "lat", "lon", 0.011)):
        c = df.groupby("region")[[la, lo]].first()
        bad = [r for r in c.index if abs(c.loc[r, la] - REGIONS[r][0]) > tol or abs(c.loc[r, lo] - REGIONS[r][1]) > tol]
        add(f"{nm}: coordinates match registry", "PASS" if not bad else "FAIL", f"mismatch={bad}")

    # dates
    p_exp = power_daily.groupby("region").date.agg(["min", "max", "nunique"])
    add("POWER: full daily coverage 2020-2025", "PASS" if (p_exp["nunique"] == n_days).all() else "WARN", p_exp["nunique"].to_dict())
    add("MODIS: composite dates on the 16-day grid", "PASS" if grid_ok(modis_raw.date.dt) else "FAIL", f"{modis_raw.date.min().date()} -> {modis_raw.date.max().date()}")
    pre = int((modis_raw.date < START_DATE).sum())
    add("MODIS: composites starting before 2020-01-01", "INFO" if pre else "PASS", f"{pre} rows (dropped: no weather coverage)")
    s_cov = (smap_daily[(smap_daily.date >= START_DATE) & (smap_daily.date <= END_DATE)].groupby("region").date.nunique() / n_days) if len(smap_daily) else pd.Series(dtype=float)
    add("SMAP: >= 90% of days 2020-2025", "PASS" if len(s_cov) == 5 and s_cov.min() >= 0.9 else "FAIL", s_cov.round(3).to_dict() if len(s_cov) else "no SMAP rows")
    # quality
    keep = modis_raw[modis_raw.reliability.isin(MODIS_KEEP_RELIABILITY)].groupby("region").size() / modis_raw.groupby("region").size()
    add(f"MODIS: >= 70% composites usable with reliability {MODIS_KEEP_RELIABILITY}", "PASS" if keep.min() >= 0.7 else "WARN", keep.round(2).to_dict())
    nd = modis_raw[modis_raw.reliability.isin(MODIS_KEEP_RELIABILITY)].ndvi
    add("MODIS: NDVI looks like vegetation (median > 0.25)", "PASS" if nd.median() > 0.25 else "WARN",
        f"median={nd.median():.3f}, p95={nd.quantile(.95):.3f} -> city-center pixels (urban/bare). Consider cropland points or an area mean.")
    add("POWER: physical ranges", "PASS" if ((power_daily.rh2m.dropna().between(0, 100)).all() and (power_daily.prec.dropna() >= 0).all()
                                         and (power_daily.t2m_max.dropna() >= power_daily.t2m_min.dropna()).all()) else "FAIL", "RH 0-100, prec >= 0, Tmax >= Tmin")
    sol = power_daily.solar.dropna()
    add("POWER: solar units", "INFO", f"mean {sol.mean():.1f}, max {sol.max():.1f} -> consistent with MJ/m2/day (kWh/m2/day would be < ~9)")
    add("SoilGrids: table status", "PASS" if soil_status == "ok" else ("WARN" if soil_status == "warn" else "FAIL"), soil_status)
    rep = pd.DataFrame(rows, columns=["status", "check", "detail"])
    display(rep)
    print("Summary:", rep.status.value_counts().to_dict())
    return rep


audit_report = run_audit()

## 4. Cleaning (NaN handling and quality filtering)

In [ ]:
# --- POWER: full daily calendar per region, short-gap interpolation of continuous variables only ---
full_days = pd.date_range(START_DATE, END_DATE)
pw = (power_daily.set_index(["region", "date"]).reindex(pd.MultiIndex.from_product([REGION_LIST, full_days], names=["region", "date"])))
meta_cols = ["lat", "lon"]
pw[meta_cols] = pw.groupby(level=0)[meta_cols].transform("first")
n_nan_before = int(pw[list(POWER_RENAME.values())].isna().sum().sum())
for c in ["t2m", "t2m_max", "t2m_min", "rh2m", "ws2m", "solar"]:
    pw[c] = pw.groupby(level=0)[c].transform(lambda s: s.interpolate(limit=3, limit_area="inside"))
pw["rh2m"] = pw["rh2m"].clip(0, 100)
pw["prec"] = pw["prec"].where(pw["prec"] >= 0)       # precipitation gaps stay NaN (never invented as 0)
pw = pw.reset_index()
print(f"POWER NaN cells: before interpolation {n_nan_before}, after {int(pw[list(POWER_RENAME.values())].isna().sum().sum())}")
sanity(pw, "POWER on full calendar", "date", "region")

# --- SMAP: full calendar, no interpolation (gaps are counted in window completeness) ---
sm = (smap_daily.set_index(["region", "date"]).reindex(pd.MultiIndex.from_product([REGION_LIST, full_days], names=["region", "date"])).reset_index())
sanity(sm, "SMAP on full calendar", "date", "region")

# --- MODIS: apply the quality policy, keep the row, blank the values that fail ---
md = modis_raw.copy()
md = md[md.date >= START_DATE].copy()
md["qa_ok"] = md.reliability.isin(MODIS_KEEP_RELIABILITY)
md.loc[~md.qa_ok, ["ndvi", "evi"]] = np.nan
md["ndvi_marginal"] = (md.reliability == 1).astype(int)
md["cycle_start"] = md["date"]
sanity(md, "MODIS after quality policy", "date", "region")
display(md.groupby("region").agg(n=("ndvi", "size"), usable=("qa_ok", "sum"), ndvi_mean=("ndvi", "mean"), ndvi_std=("ndvi", "std")).round(4))

## 5. Spatiotemporal fusion: Master Dataset (one row per region x 16-day composite)
Daily weather and SMAP are aggregated over **exactly the composite span** (`cycle_start … cycle_end`; the last composite of each year is 13–14 days long).
Aggregations: means for state variables, **sums** for flux variables (precip, ET0, GDD), **min/max** for extremes.
ET0 uses Hargreaves–Samani (needs only Tmax/Tmin and latitude, so it does not depend on the POWER radiation units).

In [ ]:
def extraterrestrial_radiation(lat_deg, doy):
    """FAO-56 eq. 21, MJ m-2 day-1."""
    phi = np.deg2rad(lat_deg); dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))


def es_kpa(t):
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))


d = pw.copy()
doy = d["date"].dt.dayofyear.values
ra = extraterrestrial_radiation(d["lat"].values, doy)
d["et0"] = np.clip(0.0023 * (d.t2m + 17.8) * np.sqrt(np.clip(d.t2m_max - d.t2m_min, 0, None)) * ra * 0.408, 0, None)   # mm/day
d["vpd"] = 0.5 * (es_kpa(d.t2m_max) + es_kpa(d.t2m_min)) * (1 - d.rh2m / 100)                                           # kPa (approx.)
d["gdd"] = ((np.minimum(d.t2m_max, 30) + np.maximum(d.t2m_min, 10)) / 2 - 10).clip(lower=0)                           # base 10, cap 30
d["hot35"] = (d.t2m_max >= 35).astype(float).where(d.t2m_max.notna())
d["frost"] = (d.t2m_min <= 2).astype(float).where(d.t2m_min.notna())
d["rainy"] = (d.prec >= 1).astype(float).where(d.prec.notna())
d["dtr"] = d.t2m_max - d.t2m_min
d["cycle_start"] = cycle_start_of(d["date"]).values
d = d[d.cycle_start >= START_DATE]

wagg = d.groupby(["region", "cycle_start"]).agg(
    t_mean=("t2m", "mean"), t_max=("t2m_max", "max"), t_min=("t2m_min", "min"), dtr_mean=("dtr", "mean"),
    rh_mean=("rh2m", "mean"), rh_min=("rh2m", "min"), vpd_mean=("vpd", "mean"), vpd_max=("vpd", "max"),
    precip_sum=("prec", "sum"), rainy_days=("rainy", "sum"), precip_max=("prec", "max"),
    wind_mean=("ws2m", "mean"), solar_mean=("solar", "mean"), et0_sum=("et0", "sum"), gdd_sum=("gdd", "sum"),
    hot_days=("hot35", "sum"), frost_days=("frost", "sum"), pw_days=("t2m", "count"), prec_days=("prec", "count")).reset_index()

s = sm.copy(); s["cycle_start"] = cycle_start_of(s["date"]).values; s = s[s.cycle_start >= START_DATE]
s = s.sort_values(["region", "date"])
sagg = s.groupby(["region", "cycle_start"]).agg(
    sms_mean=("sm_surface", "mean"), sms_min=("sm_surface", "min"), sms_max=("sm_surface", "max"),
    smrz_mean=("sm_rootzone", "mean"), smrz_min=("sm_rootzone", "min"), smrz_max=("sm_rootzone", "max"),
    smrz_first=("sm_rootzone", "first"), smrz_last=("sm_rootzone", "last"), smp_mean=("sm_profile", "mean"),
    smsw_mean=("sm_surface_wetness", "mean"), smrzw_mean=("sm_rootzone_wetness", "mean"),
    smap_days=("sm_rootzone", "count")).reset_index()
sagg["smrz_delta"] = sagg["smrz_last"] - sagg["smrz_first"]

grid = pd.concat([make_cycle_grid(2020, 2025).assign(region=r) for r in REGION_LIST], ignore_index=True)
grid["cycle_end"] = cycle_end_of(grid["cycle_start"]).values
grid["window_days"] = (grid["cycle_end"] - grid["cycle_start"]).dt.days + 1
master = (grid.merge(wagg, on=["region", "cycle_start"], how="left").merge(sagg, on=["region", "cycle_start"], how="left")
          .merge(md[["region", "cycle_start", "ndvi", "evi", "reliability", "ndvi_marginal"]], on=["region", "cycle_start"], how="left"))

# completeness: blank out a window's aggregates when too few days back them
w_cols = [c for c in wagg.columns if c not in ("region", "cycle_start", "pw_days", "prec_days")]
s_cols = [c for c in sagg.columns if c not in ("region", "cycle_start", "smap_days")]
bad_w = master.pw_days.fillna(0) < MIN_POWER_FRAC * master.window_days
bad_s = master.smap_days.fillna(0) < MIN_SMAP_FRAC * master.window_days
bad_p = master.prec_days.fillna(0) < MIN_POWER_FRAC * master.window_days
master.loc[bad_w, w_cols] = np.nan; master.loc[bad_s, s_cols] = np.nan
master.loc[bad_p, ["precip_sum", "rainy_days", "precip_max"]] = np.nan
print(f"Windows blanked for low coverage: POWER {int(bad_w.sum())}, precip {int(bad_p.sum())}, SMAP {int(bad_s.sum())}")

# MODIS dates that did not land on the grid (should be 0)
orphans = md.merge(grid[["region", "cycle_start"]], on=["region", "cycle_start"], how="left", indicator=True)
print("MODIS rows not on the composite grid:", int((orphans["_merge"] == "left_only").sum()))

# derived water / stress variables
master["p_minus_et0"] = master.precip_sum - master.et0_sum
master["aridity_ratio"] = master.precip_sum / (master.et0_sum + 1e-6)
master["sm_stress"] = 1 - master.smrzw_mean                  # 0 = wet, 1 = dry (SMAP relative wetness)

# static soil (or region one-hot fallback)
if soil_static is not None:
    soil_cols = [c for c in soil_static.columns if c.endswith("_0_30")]
    miss = soil_static[soil_cols].isna().any(axis=1)
    if miss.any():
        print("Soil NaN -> cross-region median imputation (flag soil_imputed=1) for:", soil_static.region[miss].tolist())
    soil_static = soil_static.copy(); soil_static["soil_imputed"] = miss.astype(int)
    soil_static[soil_cols] = soil_static[soil_cols].fillna(soil_static[soil_cols].median())
    master = master.merge(soil_static[["region"] + soil_cols + ["soil_imputed"]], on="region", how="left")
    STATIC_COLS = soil_cols + ["soil_imputed"]
else:
    for r in REGION_LIST:
        master["region_" + r.lower().replace(" ", "_")] = (master.region == r).astype(float)
    STATIC_COLS = [c for c in master.columns if c.startswith("region_")]
master = master.sort_values(["region", "cycle_start"]).reset_index(drop=True)

assert master.duplicated(["region", "cycle_start"]).sum() == 0
assert len(master) == 5 * 138, f"unexpected master size {len(master)}"
sanity(master, "MASTER dataset", "cycle_start", "region")
print("Static columns:", STATIC_COLS)
display(master.groupby("region")[["ndvi", "smrz_mean", "precip_sum", "et0_sum", "t_mean"]].agg(["count", "mean"]).round(3))
master.to_csv(OUT_DIR / "master_dataset.csv", index=False)

## 6. Feature engineering (causal: only cycles ≤ k are used for the origin row k)
* **Lags** of the previous 1–3 cycles for vegetation, moisture, weather and water balance.
* **Rolling windows** over the last 3 cycles (~48 days): precip, ET0, root-zone SM, NDVI.
* **Stress indices:** heat days, VPD, `sm_stress` (1 − SMAP relative wetness), 3-cycle ET0 − precip deficit, aridity ratio, GDD.
* **Vegetation gap handling:** raw `ndvi` (NaN when cloudy) plus causal `ndvi_ff` (forward-fill ≤ 3 cycles), `ndvi_age`, and quality flags.
* **Climatology anomalies** are added *after* the split from training years only (Section 7).
* **Targets:** `target_ndvi_next` and `target_smrz_next` = value at cycle k+1, found by grid position, so a cloudy gap never silently shifts the target.

In [ ]:
LAG_VARS = ["ndvi", "evi", "smrz_mean", "sms_mean", "precip_sum", "t_mean", "et0_sum", "p_minus_et0", "vpd_mean"]
DYN_FEATURES = ["t_mean", "t_max", "t_min", "dtr_mean", "rh_mean", "vpd_mean", "precip_sum", "et0_sum", "solar_mean", "wind_mean",
                "gdd_sum", "hot_days", "sms_mean", "smrz_mean", "smrz_delta", "smrzw_mean", "sm_stress", "p_minus_et0"]


def build_features(master_df):
    df = master_df.sort_values(["region", "cycle_start"]).reset_index(drop=True).copy()
    g = df.groupby("region", sort=False)
    df["pos"] = g.cumcount()
    df["ndvi_valid"] = df["ndvi"].notna().astype(int)
    df["ndvi_ff"] = g["ndvi"].transform(lambda x: x.ffill(limit=NDVI_FFILL_LIMIT))
    inv = (~df["ndvi"].notna()).astype(int)
    df["ndvi_age"] = inv.groupby([df["region"], df["ndvi_valid"].groupby(df["region"]).cumsum()]).cumsum()
    g = df.groupby("region", sort=False)
    for v in LAG_VARS:
        for k in (1, 2, 3):
            df[f"{v}_lag{k}"] = g[v].shift(k)
    for v in ("precip_sum", "et0_sum", "hot_days"):
        df[f"{v}_roll3"] = g[v].transform(lambda x: x.rolling(3, min_periods=2).sum())
    for v in ("smrz_mean", "ndvi", "t_mean", "vpd_mean"):
        df[f"{v}_roll3"] = g[v].transform(lambda x: x.rolling(3, min_periods=2).mean())
    df["water_deficit_3c"] = df["et0_sum_roll3"] - df["precip_sum_roll3"]
    df["ndvi_delta1"] = df["ndvi"] - df["ndvi_lag1"]
    df["smrz_delta1"] = df["smrz_mean"] - df["smrz_mean_lag1"]
    mid = df["cycle_start"] + pd.to_timedelta(df["window_days"] / 2, unit="D")
    df["doy_sin"] = np.sin(2 * np.pi * mid.dt.dayofyear / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * mid.dt.dayofyear / 365.25)
    df["cycle_of_year"] = ((df["cycle_start"].dt.dayofyear - 1) // 16).astype(int)
    # targets (next cycle, by grid position) and the date the target belongs to
    g = df.groupby("region", sort=False)
    df["next_cycle_start"] = g["cycle_start"].shift(-1)
    df["target_ndvi_next"] = g["ndvi"].shift(-1)
    df["target_smrz_next"] = g["smrz_mean"].shift(-1)
    df["origin_year"] = df["cycle_start"].dt.year
    df["target_year"] = df["next_cycle_start"].dt.year
    return df


feat = build_features(master)
TARGET_COLS = ["target_ndvi_next", "target_smrz_next"]
NON_FEATURE = {"region", "cycle_start", "cycle_end", "pos", "next_cycle_start", "origin_year", "target_year", "pw_days", "prec_days",
               "smap_days", "window_days", "reliability", "cycle_of_year"} | set(TARGET_COLS)
BASE_FEATURES = [c for c in feat.columns if c not in NON_FEATURE and c not in STATIC_COLS and pd.api.types.is_numeric_dtype(feat[c])]
print(f"{len(BASE_FEATURES)} base time-varying features + {len(STATIC_COLS)} static")
assert not any("next" in c for c in BASE_FEATURES), "a target-like column leaked into the features"
sanity(feat[["region", "cycle_start", "ndvi", "ndvi_ff", "ndvi_age", "smrz_mean", "smrz_mean_lag1", "precip_sum_roll3", "target_ndvi_next", "target_smrz_next"]], "FEATURE table (preview columns)", "cycle_start", "region")

### 6.1 Leakage test
Rebuild the features after **deleting every raw observation after 2024-12-31**. If any feature of a 2020–2024 row changes, that feature used future information.

In [ ]:
def leakage_test(cut="2024-12-31"):
    m2 = master.copy()
    late = m2["cycle_end"] > pd.Timestamp(cut)
    value_cols = [c for c in m2.columns if c not in ("region", "cycle_start", "cycle_end", "window_days") and c not in STATIC_COLS]
    m2.loc[late, value_cols] = np.nan                       # as if those observations had not happened yet
    f2 = build_features(m2)
    f1 = feat.copy()
    rows = (f1["cycle_end"] <= pd.Timestamp(cut)).values
    cols = BASE_FEATURES
    a, b = f1.loc[rows, cols].to_numpy(float), f2.loc[rows, cols].to_numpy(float)
    same = np.isclose(a, b, equal_nan=True)
    if not same.all():
        bad = sorted({cols[j] for j in np.where(~same.all(axis=0))[0]})
        raise AssertionError(f"Look-ahead detected in features: {bad}")
    print(f"PASS: {rows.sum()} pre-cutoff rows x {len(cols)} features are identical when the future is removed.")


leakage_test()

## 7. Time-based split (no leakage) + train-only climatology features
* **Train:** target cycle in ≤ 2024. **Test:** origin *and* target in 2025.
* The December-2024 origin has its target in January 2025, so it is in neither set (embargo).
* **Validation (early stopping only):** train rows whose target is in 2024; the test year is never used for any fit, scaling or climatology.

In [ ]:
TASKS = {"target_ndvi_next": {"ref": "ndvi_ff", "label": "NDVI(k+1)"},
         "target_smrz_next": {"ref": "smrz_mean", "label": "Root-zone SM(k+1) [m3/m3]"}}

is_train = (feat.target_year <= TEST_YEAR - 1)
is_test = (feat.origin_year >= TEST_YEAR) & (feat.target_year <= TEST_YEAR)
is_val = is_train & (feat.target_year == VAL_YEAR)
is_inner = is_train & (feat.target_year < VAL_YEAR)
assert not (is_train & is_test).any()
assert feat.loc[is_train, "next_cycle_start"].max() < feat.loc[is_test, "cycle_start"].min(), "train targets overlap the test period"
print("Embargoed origin rows (target falls in test year):", int(((feat.origin_year < TEST_YEAR) & (feat.target_year >= TEST_YEAR)).sum()))


def add_climatology(df, train_mask):
    """Region x cycle-of-year climatology computed from TRAIN rows only -> anomalies and 'next cycle' climatology."""
    out = df.copy()
    # ndvi / smrz at the cycle itself, from train-period cycles (cycle_end <= last train date)
    tr = out[train_mask | (out.cycle_start < pd.Timestamp(f"{TEST_YEAR}-01-01"))]
    clim = tr.groupby(["region", "cycle_of_year"]).agg(ndvi_clim=("ndvi", "mean"), smrz_clim=("smrz_mean", "mean")).reset_index()
    reg_mean = tr.groupby("region").agg(ndvi_rm=("ndvi", "mean"), smrz_rm=("smrz_mean", "mean")).reset_index()
    out = out.merge(clim, on=["region", "cycle_of_year"], how="left").merge(reg_mean, on="region", how="left")
    out["ndvi_clim"] = out["ndvi_clim"].fillna(out["ndvi_rm"]); out["smrz_clim"] = out["smrz_clim"].fillna(out["smrz_rm"])
    out["ndvi_anom"] = out["ndvi"] - out["ndvi_clim"]; out["smrz_anom"] = out["smrz_mean"] - out["smrz_clim"]
    nxt = clim.copy(); nxt["cycle_of_year"] = (nxt["cycle_of_year"] - 1) % 23
    nxt = nxt.rename(columns={"ndvi_clim": "ndvi_clim_next", "smrz_clim": "smrz_clim_next"})
    out = out.merge(nxt, on=["region", "cycle_of_year"], how="left")
    return out.drop(columns=["ndvi_rm", "smrz_rm"])


feat = feat.reset_index(drop=True)
feat2 = add_climatology(feat, is_train.values)
assert (feat2["region"].values == feat["region"].values).all() and (feat2["cycle_start"].values == feat["cycle_start"].values).all()
for c in ("ndvi_clim", "smrz_clim", "ndvi_anom", "smrz_anom", "ndvi_clim_next", "smrz_clim_next"):
    feat[c] = feat2[c].values
FEATURES = BASE_FEATURES + ["ndvi_anom", "smrz_anom"] + STATIC_COLS
print("Total model features:", len(FEATURES))
for k, name in (("Train", is_train), ("  inner-train", is_inner), ("  validation", is_val), ("Test", is_test)):
    sub = feat[name]
    print(f"{k:14s} rows={len(sub):4d}  origin {sub.cycle_start.min().date()} -> {sub.cycle_start.max().date()}  target up to {sub.next_cycle_start.max().date()}")

## 8. Evaluation utilities and baselines
Every model is scored on the **same 2025 rows** against two reference forecasts:
* **Persistence:** next cycle = current value (hard to beat for slowly varying soil moisture).
* **Climatology:** next cycle = train-years mean for that region and cycle of year.

Models predict `next − current` and the current value is added back, so RMSE/MAE/R² are on the original scale.

In [ ]:
import copy


def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"RMSE": float(np.sqrt(mean_squared_error(y, p))), "MAE": float(mean_absolute_error(y, p)),
            "R2": float(r2_score(y, p)) if len(y) > 1 else float("nan")}


def eligible(target):
    ref = TASKS[target]["ref"]
    return (feat[target].notna() & feat[ref].notna() & (feat["pos"] >= SEQ_LEN - 1)).values


RESULTS, PREDS, IMPORTANCE = {}, {}, {}
for tgt in TASKS:
    el = eligible(tgt); te = el & is_test.values; tr = el & is_train.values; va = el & is_val.values; ii = el & is_inner.values
    print(f"{tgt}: eligible={el.sum()} | inner-train={ii.sum()} val={va.sum()} train-all={tr.sum()} test={te.sum()}")
    assert te.sum() > 20 and ii.sum() > 50, "too few rows to train/evaluate"
    ref = TASKS[tgt]["ref"]; y_te = feat.loc[te, tgt].values
    base = {"Persistence": feat.loc[te, ref].values,
            "Climatology": feat.loc[te, "ndvi_clim_next" if "ndvi" in tgt else "smrz_clim_next"].values}
    PREDS[tgt] = {"idx": np.where(te)[0], "actual": y_te, **base}
    RESULTS[tgt] = {k: metrics(y_te, v) for k, v in base.items()}
    print(f"  baselines on test: " + " | ".join(f"{k} RMSE={v['RMSE']:.4f} R2={v['R2']:.3f}" for k, v in RESULTS[tgt].items()))

## 9. Baseline model: XGBoost (with feature-importance ranking)

In [ ]:
XGB_PARAMS = dict(n_estimators=2000, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
                  reg_lambda=2.0, objective="reg:squarederror", random_state=SEED, n_jobs=-1, tree_method="hist")


def fit_xgb(tgt):
    ref = TASKS[tgt]["ref"]; el = eligible(tgt)
    ii, va, tr, te = (el & m.values for m in (is_inner, is_val, is_train, is_test))
    X = feat[FEATURES]; y = feat[tgt] - feat[ref]
    m = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=50)
    m.fit(X[ii], y[ii], eval_set=[(X[va], y[va])], verbose=False)
    best = int(m.best_iteration) + 1
    final = xgb.XGBRegressor(**{**XGB_PARAMS, "n_estimators": best}).fit(X[tr], y[tr])
    pred = feat.loc[te, ref].values + final.predict(X[te])
    gain = pd.Series(final.get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0.0)
    return final, pred, best, (gain / gain.sum()).sort_values(ascending=False)


XGB_MODELS = {}
for tgt in TASKS:
    model, pred, best, imp = fit_xgb(tgt)
    XGB_MODELS[tgt] = model; PREDS[tgt]["XGBoost"] = pred; IMPORTANCE[tgt] = imp
    RESULTS[tgt]["XGBoost"] = metrics(PREDS[tgt]["actual"], pred)
    assert np.isfinite(pred).all()
    print(f"\n{tgt}: best iteration {best} | test {RESULTS[tgt]['XGBoost']}")
    print("  top-10 features by gain share:"); display(imp.head(10).round(4).to_frame("gain_share"))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, tgt in zip(axes, TASKS):
    top = IMPORTANCE[tgt].head(20)[::-1]
    ax.barh(top.index, top.values, color="#2a7f62"); ax.set_title(f"XGBoost feature importance (gain) — {TASKS[tgt]['label']}")
plt.tight_layout(); plt.show()

## 10. Hybrid deep model: multi-branch GRU
* **Temporal branch:** GRU over the last `SEQ_LEN` cycles of weather, SMAP and gap-filled NDVI (+ flags, seasonality).
* **Static branch:** MLP over SoilGrids features (or region one-hot if SoilGrids is unavailable).
* Fused head predicts the standardized `next − current`. Training: AdamW, dropout, early stopping on the 2024 validation year,
  then refit on all training rows for the best epoch count; **ensemble of `N_SEEDS` seeds**.
* Scalers, medians and fills use pre-2025 data only.

In [ ]:
SEQ_DYN = DYN_FEATURES + ["ndvi_ff", "ndvi_valid", "ndvi_marginal", "doy_sin", "doy_cos"]


class HybridGRU(nn.Module):
    def __init__(self, n_dyn, n_static, hid=32, st_hid=16, drop=0.25):
        super().__init__()
        self.gru = nn.GRU(n_dyn, hid, batch_first=True)
        self.static = nn.Sequential(nn.Linear(n_static, st_hid), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Sequential(nn.Linear(hid + st_hid, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))

    def forward(self, x_seq, x_static):
        _, h = self.gru(x_seq)
        return self.head(torch.cat([h[-1], self.static(x_static)], dim=1))


def prepare_arrays():
    d = feat
    pre_test = (d.cycle_start < pd.Timestamp(f"{TEST_YEAR}-01-01")).values
    dyn = d[SEQ_DYN].copy()
    dyn = dyn.groupby(d["region"]).ffill(limit=2)                 # causal fill within a region
    med = dyn[pre_test].median().fillna(0.0)
    dyn = dyn.fillna(med)
    mu, sd = dyn[pre_test].mean(), dyn[pre_test].std().replace(0, 1.0).fillna(1.0)
    dyn_arr = ((dyn - mu) / sd).to_numpy(np.float32)
    reg_tab = d.drop_duplicates("region")[STATIC_COLS]
    smu, ssd = reg_tab.mean(), reg_tab.std(ddof=0).replace(0, 1.0).fillna(1.0)
    st_arr = ((d[STATIC_COLS] - smu) / ssd).to_numpy(np.float32)
    return dyn_arr, st_arr


DYN_ARR, ST_ARR = prepare_arrays()
assert np.isfinite(DYN_ARR).all() and np.isfinite(ST_ARR).all()
print("Dynamic matrix:", DYN_ARR.shape, "| static matrix:", ST_ARR.shape, "| sequence length:", SEQ_LEN)


def seq_batch(rows):
    idx = rows[:, None] + np.arange(-SEQ_LEN + 1, 1)[None, :]
    return DYN_ARR[idx], ST_ARR[rows]


def _t(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)


def train_gru(rows_tr, y_tr, rows_va=None, y_va=None, epochs=300, patience=30, fixed=None, seed=0, lr=3e-3, wd=1e-3, bs=32):
    torch.manual_seed(seed); np.random.seed(seed)
    Xs, Xt = (_t(a) for a in seq_batch(rows_tr)); y = _t(y_tr).unsqueeze(1)
    model = HybridGRU(Xs.shape[2], Xt.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd); loss_fn = nn.MSELoss()
    if rows_va is not None:
        Vs, Vt = (_t(a) for a in seq_batch(rows_va)); Vy = _t(y_va).unsqueeze(1)
    best, best_ep, best_state, bad, hist = float("inf"), 0, None, 0, []
    for ep in range(1, (fixed or epochs) + 1):
        model.train(); perm = torch.randperm(len(y), device=DEVICE)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            opt.zero_grad(); loss = loss_fn(model(Xs[b], Xt[b]), y[b]); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if rows_va is not None and fixed is None:
            model.eval()
            with torch.no_grad():
                vl = loss_fn(model(Vs, Vt), Vy).item()
            hist.append(vl)
            if vl < best - 1e-5:
                best, best_ep, bad, best_state = vl, ep, 0, copy.deepcopy(model.state_dict())
            else:
                bad += 1
                if bad >= patience:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, hist


def predict_gru(model, rows):
    model.eval(); xs, xt = (_t(a) for a in seq_batch(rows))
    with torch.no_grad():
        return model(xs, xt).squeeze(1).cpu().numpy()


GRU_STATE, GRU_HIST = {}, {}
t0 = time.time()
for tgt in TASKS:
    ref = TASKS[tgt]["ref"]; el = eligible(tgt)
    ii, va, tr, te = (np.where(el & m.values)[0] for m in (is_inner, is_val, is_train, is_test))
    delta = (feat[tgt] - feat[ref]).to_numpy(float)
    ymu, ysd = delta[tr].mean(), delta[tr].std() or 1.0
    z = lambda rows: (delta[rows] - ymu) / ysd
    preds, bests, states = [], [], []
    for s in range(N_SEEDS):
        _, best_ep, hist = train_gru(ii, z(ii), va, z(va), seed=SEED + s)
        final, _, _ = train_gru(tr, z(tr), fixed=max(best_ep, 10), seed=SEED + s)
        preds.append(predict_gru(final, te) * ysd + ymu); bests.append(best_ep); states.append(final.state_dict())
        if s == 0: GRU_HIST[tgt] = hist
    pred = feat.loc[feat.index[te], ref].values + np.mean(preds, axis=0)
    assert np.isfinite(pred).all() and len(pred) == len(PREDS[tgt]["actual"])
    PREDS[tgt]["GRU-Hybrid"] = pred; GRU_STATE[tgt] = states[0]
    RESULTS[tgt]["GRU-Hybrid"] = metrics(PREDS[tgt]["actual"], pred)
    print(f"{tgt}: best epochs per seed {bests} | test {RESULTS[tgt]['GRU-Hybrid']} | {time.time()-t0:.0f}s")

## 11. Test-set results (2025), plots and artifacts

In [ ]:
rows = []
for tgt in TASKS:
    for model_name, m in RESULTS[tgt].items():
        rows.append({"target": TASKS[tgt]["label"], "model": model_name, "n_test": len(PREDS[tgt]["actual"]), **m})
metrics_df = pd.DataFrame(rows)
print("=" * 78, "\nTEST SET (2025) — RMSE / MAE / R²\n" + "=" * 78)
for tgt in TASKS:
    print("\n" + TASKS[tgt]["label"]); display(metrics_df[metrics_df.target == TASKS[tgt]["label"]].drop(columns="target").round(4).reset_index(drop=True))

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for r_i, tgt in enumerate(TASKS):
    P = PREDS[tgt]; reg = feat.loc[P["idx"], "region"].values
    for c_i, mname in enumerate(["XGBoost", "GRU-Hybrid"]):
        ax = axes[r_i, c_i]
        for rg in REGION_LIST:
            mk = reg == rg; ax.scatter(P["actual"][mk], P[mname][mk], s=22, label=rg, alpha=.8)
        lo, hi = np.nanmin(P["actual"]), np.nanmax(P["actual"]); ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_title(f"{mname} — {TASKS[tgt]['label']}  (R²={RESULTS[tgt][mname]['R2']:.2f})"); ax.set_xlabel("actual"); ax.set_ylabel("predicted")
        if r_i == 0 and c_i == 0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(len(REGION_LIST), 2, figsize=(16, 3 * len(REGION_LIST)), sharex="col")
for c_i, tgt in enumerate(TASKS):
    P = PREDS[tgt]; dfp = pd.DataFrame({"date": feat.loc[P["idx"], "next_cycle_start"].values, "region": feat.loc[P["idx"], "region"].values,
                                        "actual": P["actual"], "Persistence": P["Persistence"], "XGBoost": P["XGBoost"], "GRU-Hybrid": P["GRU-Hybrid"]})
    for r_i, rg in enumerate(REGION_LIST):
        ax = axes[r_i, c_i]; sub = dfp[dfp.region == rg]
        ax.plot(sub.date, sub.actual, "k-o", ms=3, label="actual"); ax.plot(sub.date, sub["Persistence"], ":", color="grey", label="persistence")
        ax.plot(sub.date, sub["XGBoost"], "-", label="XGBoost"); ax.plot(sub.date, sub["GRU-Hybrid"], "-", label="GRU")
        ax.set_title(f"{rg} — {TASKS[tgt]['label']}", fontsize=9)
        if r_i == 0 and c_i == 0: ax.legend(fontsize=7, ncol=4)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, tgt in zip(axes, TASKS):
    ax.plot(GRU_HIST[tgt]); ax.set_title(f"GRU validation loss (seed 0) — {TASKS[tgt]['label']}"); ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()

print("\nPer-region RMSE on test (XGBoost vs GRU vs persistence):")
for tgt in TASKS:
    P = PREDS[tgt]; reg = feat.loc[P["idx"], "region"].values
    tab = pd.DataFrame({m: [np.sqrt(np.mean((P["actual"][reg == rg] - P[m][reg == rg]) ** 2)) for rg in REGION_LIST]
                        for m in ("Persistence", "XGBoost", "GRU-Hybrid")}, index=REGION_LIST)
    print(TASKS[tgt]["label"]); display(tab.round(4))

# artifacts
metrics_df.to_csv(OUT_DIR / "metrics_test_2025.csv", index=False)
pd.DataFrame(IMPORTANCE).to_csv(OUT_DIR / "xgb_feature_importance.csv")
feat.to_csv(OUT_DIR / "features_table.csv", index=False)
for tgt in TASKS:
    XGB_MODELS[tgt].save_model(str(OUT_DIR / f"xgb_{tgt}.json")); torch.save(GRU_STATE[tgt], OUT_DIR / f"gru_{tgt}.pt")
    P = PREDS[tgt]; pd.DataFrame({"region": feat.loc[P["idx"], "region"].values, "origin": feat.loc[P["idx"], "cycle_start"].values,
                                  "target_cycle": feat.loc[P["idx"], "next_cycle_start"].values, **{k: P[k] for k in ("actual", "Persistence", "Climatology", "XGBoost", "GRU-Hybrid")}}
                 ).to_csv(OUT_DIR / f"predictions_{tgt}.csv", index=False)
print("\nSaved to", OUT_DIR.resolve(), ":", sorted(p.name for p in OUT_DIR.iterdir()))
if IN_COLAB and not DEMO_MODE and os.path.isdir("/content/drive/MyDrive"):
    shutil.copytree(OUT_DIR, "/content/drive/MyDrive/SpaceSoil/outputs", dirs_exist_ok=True); print("Copied outputs to Drive/SpaceSoil/outputs")

## 12. How to read these results (read before trusting any number)
* **Compare every model with *Persistence* and *Climatology*.** An R² above 0.9 on soil moisture is usually just persistence.
* **Only 5 regions.** SoilGrids is constant per region, so the static branch is effectively a region ID. The test shows generalization **over time**, not to new places.
* **~690 samples, 115 test rows:** metric differences of a few percent are within noise; the seed ensemble reduces but does not remove it.
* **NDVI from city-center pixels** (~0.13) has very little seasonal signal; if NDVI R² is low, that is a data-validity limit, not a modeling bug.
* **`DEMO_MODE=True` output is synthetic** and only proves that the pipeline executes without leakage.